# 05 — A demanding case: spam (where AdaBoost shines, and where label noise bites)

*Chapter 07 · Notebook 5 of 5*

Welcome to the capstone. The whole chapter converges here. You built AdaBoost by
hand (NB 1), saw where its weights come from (NB 2), watched it resist overfitting
on clean data and stumble on noisy data (NB 3), and turned the real estimator's
dials (NB 4). Now we put it on a real, high-dimensional problem and ask two honest
questions: **does AdaBoost shine here, and can we tell its noise story truthfully?**

**Prerequisites.** NB 1–4 (reweighting by hand and the by-hand/sklearn parity; the
additive model and exponential loss; rounds x learning_rate and overfitting; the
estimator `AdaBoostClassifier` and its dials, the base must stay weak). The
cross-method spine: ch 01 KNN, ch 03 logistic regression, ch 04 decision tree,
ch 05 SVM, ch 06 random forest. From module 00: the confusion matrix, precision
and recall (NB 07) and the train/test split (NB 04).

**What you will be able to do.** Run an honest AdaBoost workflow on a real
classification problem; see it turn a weak stump into a strong classifier; read
its staged resistance curve; read its feature importances with the right caveats;
understand its noise behaviour *from the inside* (which points hoard the weight);
and refuse to over-generalise a tidy-looking comparison into a law it does not earn.


## Where we are, and the stakes

**spambase** is the dataset boosting grew up on (Hastie, Tibshirani & Friedman,
*The Elements of Statistical Learning*, ch 10). It holds **4601 emails**, each
described by **57 hand-crafted features**: the frequency of 48 words, the frequency
of 6 punctuation characters, and 3 summaries of runs of capital letters. The label
is `spam` / `not-spam`.

This is a genuine workhorse problem: real, moderately sized, fully numeric, mildly
imbalanced. A single decision stump (NB 1) barely beats guessing on it. By the end
of this notebook a few hundred stacked stumps will be among the strongest models on
the table — and we will have looked honestly at exactly where that strength comes
from, and where it runs out.


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

# Show the one-time download from OpenML (never silence console output).
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
viz.use_course_style()

SEED = 0

# Fetch spambase as a named DataFrame + Series (pandas-first; no loader wrapper needed).
spam = fetch_openml("spambase", version=1, as_frame=True)
X = spam.data
y = spam.target.astype(int).to_numpy()  # OpenML labels are the strings "0"/"1"
y_named = pd.Series(np.where(y == 1, "spam", "not-spam"), name="label")

# Feature groups, by name prefix (already descriptive in OpenML).
word_cols = [c for c in X.columns if c.startswith("word_freq_")]
char_cols = [c for c in X.columns if c.startswith("char_freq_")]
caps_cols = [c for c in X.columns if c.startswith("capital_run_length_")]

print(f"spambase: {X.shape[0]} emails x {X.shape[1]} features")
print(f"  word-frequency features: {len(word_cols)}")
print(f"  char-frequency features: {len(char_cols)}")
print(f"  capital-run-length features: {len(caps_cols)}")
print(f"class balance: spam {(y == 1).mean():.3f}  not-spam {(y == 0).mean():.3f}")

# One 70/30 stratified split, fixed seed. No scaling for AdaBoost / RF / trees;
# the distance/linear/margin baselines get a StandardScaler in their pipelines.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
Xtr, Xte = X_train.to_numpy(), X_test.to_numpy()
print(f"train {len(y_train)}   test {len(y_test)}   (stratified, seed {SEED})")


In [ ]:
# Figure A — how many emails of each class
fig = viz.plot_class_balance(y_named)
fig.suptitle("spambase — class balance (all 4601 emails)")
plt.show()


**Read the figure (A).** Spam is **39.4 %** of the mail, not-spam **60.6 %** — a
*mild* imbalance, nothing like the 200-to-1 rare classes of chapter 06's forest
cover types. So plain accuracy is a reasonable headline here. We will still open the
confusion matrix later, because the two mistakes do not cost the same: flagging a
real email as spam (a false alarm, lost mail) usually hurts more than letting one
spam through.


## Does AdaBoost shine here? An honest cross-method comparison

The honest workflow is the one the course has used since chapter 01: fit a spread of
model families on the *same* training set, score them once on the *same* sealed test
set, and let the numbers speak. AdaBoost, the random forest, the single tree and the
lone stump are tree-based, so they need **no scaling** (NB 4). KNN, logistic
regression and the SVM measure distances or dot products, so each gets a
`StandardScaler` (the chapter-01 scale trap). The real question is not "what is the
best number" but **which model *family* fits the shape of this problem.**


In [ ]:
# Fit one model per family on the same split; score once on the sealed test set.
models = {
    "stump": DecisionTreeClassifier(max_depth=1, random_state=SEED),
    "tree (full)": DecisionTreeClassifier(random_state=SEED),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(random_state=SEED)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=400, random_state=SEED),
}
acc = {}
for name, model in models.items():
    model.fit(Xtr, y_train)
    acc[name] = model.score(Xte, y_test)
    print(f"  {name:14s} test accuracy = {acc[name]:.4f}")

# Keep the two fitted ensembles for the staged / confusion figures below.
ada400 = models["AdaBoost"]

# Figure B — cross-method test accuracy (AdaBoost and the forest highlighted).
order = sorted(acc, key=acc.get)
bar_colors = [
    COLORS["highlight"] if k == "AdaBoost"
    else COLORS["model"] if k == "RandomForest"
    else COLORS["muted"]
    for k in order
]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(order, [acc[k] for k in order], color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
for i, k in enumerate(order):
    ax.text(acc[k] - 0.004, i, f"{acc[k]:.3f}", va="center", ha="right", color=COLORS["text"])
ax.set_xlim(0.70, 0.98)
ax.set_xlabel("test accuracy")
ax.set_title("spambase — accuracy by model family (AdaBoost in rose, forest in blue)")
plt.show()


**Read the figure (B).** AdaBoost reaches **0.949** and the random forest **0.959** —
the two ensembles lead the table. They beat the single full tree (0.915), the linear
and margin and distance methods (logistic regression 0.926, SVM 0.936, KNN 0.899),
and they tower over a lone stump (**0.782**). That last gap is the chapter's whole
promise made concrete: AdaBoost took a weak stump that is barely better than guessing
and, by stacking 400 of them, turned it into a 0.949 classifier.

AdaBoost is **competitive with** the forest, which edges it by one point — not a win,
and we will not pretend it is one. There is no universally best model, and the contrast
with chapter 05 makes the point cleanly: on breast-cancer — a near-linear problem once
the features are scaled — the **tuned SVM** led the four methods compared there
(0.965); here, on spam, the tree **ensembles** lead. The right tool tracks the *shape*
of the problem, and spam's shape (many weakly-informative token counts, no clean linear
boundary) suits an ensemble of trees.


## Resistance on real data: do more rounds overfit?

Chapter 3 showed AdaBoost's famous resistance to overfitting on clean data: the test
error keeps falling, or holds a flat band, even after the training error has bottomed
out. One well-supported explanation is that boosting keeps widening its margins long
after it has fit the data (Schapire et al. 1998). We measured that on a 2-D toy. Does
it hold on a real 57-feature problem?
We read the **staged** test and training error: the score of the partial ensemble
after each of the 400 rounds.


In [ ]:
# staged_score replays the ensemble after each round (the partial-vote score).
rounds = np.arange(1, 401)
test_err = 1.0 - np.fromiter(ada400.staged_score(Xte, y_test), dtype=float)
train_err = 1.0 - np.fromiter(ada400.staged_score(Xtr, y_train), dtype=float)

bottom = int(np.argmin(test_err))
drift = test_err[-1] - test_err[bottom]
print(f"test error @ round 1   : {test_err[0]:.4f}")
print(f"test error bottoms     : {test_err[bottom]:.4f} @ round {bottom + 1}")
print(f"test error @ round 400 : {test_err[-1]:.4f}   (drift +{drift:.4f})")
print(f"train error floor      : {train_err.min():.4f}   (reaches zero: {train_err.min() == 0})")

# Figure C — staged resistance curve (sub-sampled rounds keep the markers readable).
keep = np.arange(0, 400, 10)
fig = viz.plot_train_test_curve(
    rounds[keep], train_err[keep], test_err[keep],
    xlabel="number of rounds (weak learners)", ylabel="error",
)
fig.axes[0].axvline(bottom + 1, color=COLORS["muted"], linestyle="--", linewidth=1)
fig.axes[0].set_title("spambase — AdaBoost staged error (clean data): resistance")
plt.show()


**Read the figure (C).** The test error falls from 0.218 (one stump) to a minimum of
**0.0485 at round 279**, then holds essentially flat to 0.0507 at round 400 — a drift
of two-thousandths over the last 120 rounds, with no upward climb. This is the
textbook resistance curve, now on real data: piling on rounds does not overfit clean,
well-separated data.

One honest correction to the toy intuition: the **training error floors near 0.045
and never reaches zero**. On the 2-D moons of NB 2–3 the training error marched to
exactly 0; that was a property of a clean, low-noise toy. Real data carries
irreducible overlap (some spam and some ham look alike in these 57 features), so even
an infinitely deep boosting run cannot drive training error to zero. The resistance is
real — and we are about to see it is *not* immunity.


In [ ]:
# Figure D — where the 400-round AdaBoost makes its mistakes.
cm = confusion_matrix(y_test, ada400.predict(Xte))
tn, fp, fn, tp = cm.ravel()
print(f"confusion matrix (rows = true, cols = predicted):\n{cm}")
print(f"spam recall    = {tp / (tp + fn):.3f}   ({fn} spam emails missed)")
print(f"spam precision = {tp / (tp + fp):.3f}   ({fp} real emails wrongly flagged)")

fig = viz.plot_confusion_matrix(cm, ["not-spam", "spam"])
fig.suptitle("spambase — AdaBoost(400) confusion matrix")
plt.show()


**Read the figure (D).** Of 544 real spam emails, AdaBoost catches 501 and **misses
43** (recall 0.921); of the emails it labels spam, **27 are actually legitimate**
(precision 0.949). The two errors are not symmetric in cost: those 27 false alarms
are real messages diverted to a junk folder — for a spam filter, usually the more
painful mistake. The model reports a single default-threshold decision here; chapter
03's threshold-moving is the lever you would reach for to trade recall against
precision, and you would trust it only because the costs differ, not because 0.5 is
wrong.


## Reading the importances honestly

Like any tree ensemble, AdaBoost exposes `feature_importances_`: the impurity
decrease (MDI) each feature brought, averaged over the stumps and weighted by their
vote alpha. Chapter 04 and chapter 06 warned that MDI is biased and not causal, and
named **permutation importance** (shuffle one column, measure the held-out accuracy
drop) as the honest cross-check. Chapter 06 put it to work on cover types; we keep
that promise here, on 57 real features.


In [ ]:
# A lighter 200-round model for importances (permutation refits nothing, only re-scores).
ada200 = AdaBoostClassifier(n_estimators=200, random_state=SEED).fit(Xtr, y_train)
mdi = ada200.feature_importances_
perm = permutation_importance(ada200, Xte, y_test, n_repeats=5, random_state=SEED)
print(f"features with non-zero MDI: {(mdi > 0).sum()} / {len(mdi)}")


def pretty(name):
    # OpenML URL-encodes the punctuation char-frequency columns; decode for display.
    repl = {"%21": "!", "%24": "$", "%23": "#", "%28": "(", "%3B": ";", "%5B": "["}
    out = name.replace("word_freq_", "word:").replace("char_freq_", "char:")
    for code_, ch in repl.items():
        out = out.replace(code_, ch)
    return out.replace("capital_run_length_", "caps:")


names = [pretty(c) for c in X.columns]

# Figure E — MDI (left) vs permutation importance (right), top 12 each.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_feature_importances(names, mdi, top=12, label="MDI", ax=axes[0])
axes[0].set_title("impurity importance (MDI)")
viz.plot_feature_importances(
    names, perm.importances_mean, std=perm.importances_std, top=12,
    label="permutation", color=COLORS["correct"], ax=axes[1],
)
axes[1].set_title("permutation importance (held-out drop)")
fig.suptitle("spambase — AdaBoost feature importance: MDI vs permutation")
fig.tight_layout()
plt.show()


**Read the figure (E).** Both rankings surface the same kind of leaders, and they fall
into two honest groups. The first is a set of **genuine spam markers**: the
characters `!` and `$`, and words like `remove`, `free`, `your` — exactly the tokens
you would expect a junk filter to lean on. The second is a set of **corpus
artifacts**: `george`, `hp`, `edu`, `meeting`, `cs`, `1999`. spambase was collected
at Hewlett-Packard Labs from the mailbox of an employee named George, so those tokens
are strong markers of *legitimate* mail — for this one person, in 1999. MDI and
permutation **agree on who the leaders are but reorder them**, and only 36 of the 57
features carry any MDI at all.

The honest lesson is the one chapter 04 promised: importance describes **this model on
this corpus**, and it is **not causal**. `george` is not a general property of
non-spam email; it is an accident of where the data came from. Read importances at the
semantic level, cross-check MDI against permutation, and never ship a model on the
strength of a feature ranking alone.


## The noise story, from the inside

Chapter 3 showed that label noise can make AdaBoost overfit: with mislabeled training
points, more rounds eventually *raise* the test error. Here is *why*, and it traces
straight back to NB 2. AdaBoost minimises the **exponential loss**, which punishes a
confidently-wrong prediction ever more harshly. So the reweighting keeps pouring
weight onto the points the ensemble cannot get right — and a **mislabeled** point is,
by construction, a point it cannot get right. We will measure that pile-up by hand,
watch what it does to the test error, and then refuse to turn it into a slogan.


In [ ]:
# Inject 20% label noise into the TRAINING set only; the test set stays clean.
rng = np.random.default_rng(SEED)
flip = rng.random(len(y_train)) < 0.20
y_noisy = y_train.copy()
y_noisy[flip] = 1 - y_noisy[flip]
print(f"flipped {flip.sum()} train labels ({flip.mean():.3f} of {len(y_train)})")

# Run AdaBoost by hand (SAMME) and track the weight held by the flipped points.
w = np.ones(len(y_train)) / len(y_train)
share = []
for _ in range(200):
    stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
    stump.fit(Xtr, y_noisy, sample_weight=w)
    miss = stump.predict(Xtr) != y_noisy
    eps = float(np.clip(np.dot(w, miss), 1e-12, 1 - 1e-12))
    alpha = np.log((1 - eps) / eps)  # SAMME, two classes (+ ln(K-1) = 0)
    w *= np.exp(alpha * miss)
    w /= w.sum()
    share.append(w[flip].sum())
share = np.array(share)
print(f"weight on the {flip.mean():.0%} mislabeled points: "
      f"{share[0]:.3f} (round 1) -> {share[-1]:.3f} (round 200)")

# Staged test error on the clean test set: clean run vs a heavy 40% flip.
rng = np.random.default_rng(SEED)
flip40 = rng.random(len(y_train)) < 0.40
y40 = y_train.copy()
y40[flip40] = 1 - y40[flip40]
ada40 = AdaBoostClassifier(n_estimators=400, random_state=SEED).fit(Xtr, y40)
err40 = 1.0 - np.fromiter(ada40.staged_score(Xte, y_test), dtype=float)
b40 = int(np.argmin(err40))
print(f"40% noise: test error bottoms {err40[b40]:.4f} @ round {b40 + 1}, "
      f"climbs to {err40[-1]:.4f} @ round 400 (+{err40[-1] - err40[b40]:.4f})")

# Figure F — left: weight concentration; right: clean vs noisy staged test error.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(np.arange(1, 201), share, color=COLORS["error"], linewidth=2)
axes[0].axhline(flip.mean(), color=COLORS["muted"], linestyle="--", linewidth=1,
                label=f"their share of points ({flip.mean():.0%})")
axes[0].set_xlabel("number of rounds")
axes[0].set_ylabel("weight held by the mislabeled points")
axes[0].set_ylim(0, 0.5)
axes[0].legend()
axes[0].set_title("20% noise: weight piles onto the mislabeled points")
axes[1].plot(rounds[keep], test_err[keep], color=COLORS["correct"], linewidth=2, label="clean")
axes[1].plot(rounds[keep], err40[keep], color=COLORS["error"], linewidth=2, label="40% label noise")
axes[1].axvline(b40 + 1, color=COLORS["muted"], linestyle="--", linewidth=1)
axes[1].set_xlabel("number of rounds")
axes[1].set_ylabel("test error (clean test set)")
axes[1].legend()
axes[1].set_title("noise turns resistance into overfitting")
fig.tight_layout()
plt.show()


**Read the figure (F).** *Left:* the mislabeled points are about 21 % of the training
set, but within a handful of rounds they come to hold roughly **40 % of the total
weight** — nearly double their share — and they stay there. This is the exponential
loss of NB 2 made visible: it chases the points it cannot fit, and the mislabeled ones
are exactly those. *Right:* at this scale spam still largely **resists** modest noise
(the 20 % run, not shown, drifts up only ~0.005 over 400 rounds), but under a heavy
**40 %** flip the test error bottoms early — around round 40 — and then **climbs**
(+0.035 by round 400). Past the bottom, the extra rounds are spending their effort
memorising noise.

The levers are the ones NB 3 named: **early stopping** (stop near the bottom), a
smaller `learning_rate` (slower, smoother fitting), or a loss less savage than the
exponential (chapter 08). Resistance is real, but it is **not immunity**.


## Is "the forest beats AdaBoost under noise" a law?

There is a tidy piece of folklore: boosting is fragile to label noise, bagging is
robust, so a random forest should beat AdaBoost as noise grows. It is the kind of
clean story that is easy to repeat. Let us test it honestly on spam — fit both at
several noise levels and compare on the clean test set.


In [ ]:
# Degradation curve: AdaBoost(400) vs RandomForest(300) as train-label noise grows.
levels = [0.0, 0.1, 0.2, 0.3, 0.4]
ada_acc, rf_acc = [], []
for nz in levels:
    rng = np.random.default_rng(SEED)
    fl = rng.random(len(y_train)) < nz
    yn = y_train.copy()
    yn[fl] = 1 - yn[fl]
    a = AdaBoostClassifier(n_estimators=400, random_state=SEED).fit(Xtr, yn)
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(Xtr, yn)
    ada_acc.append(a.score(Xte, y_test))
    rf_acc.append(rf.score(Xte, y_test))
    print(f"noise {nz:.1f}:  AdaBoost {ada_acc[-1]:.4f}   RandomForest {rf_acc[-1]:.4f}")

# Figure G — the two degradation curves.
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(levels, ada_acc, color=COLORS["highlight"], marker="o", linewidth=2, label="AdaBoost")
ax.plot(levels, rf_acc, color=COLORS["model"], marker="s", linewidth=2, label="RandomForest")
ax.set_xlabel("fraction of training labels flipped")
ax.set_ylabel("test accuracy (clean test set)")
ax.legend()
ax.set_title("spambase — degradation under label noise")
plt.show()


**Read the figure (G).** On clean data the forest edges AdaBoost (0.959 vs 0.949). But
from **20 % noise upward the ranking flips**: AdaBoost is the *more* robust of the
two (at 30 %: 0.897 vs 0.839; at 40 %: 0.823 vs 0.704, a 12-point gap). The folklore
**reverses** on spam.

Why? It is the lesson of NB 4, turned around. AdaBoost's base learner is a **weak
stump** — far too simple to carve out and memorise an individual mislabeled point.
The random forest grows **deep** trees on purpose, and a deep tree has exactly the
capacity to fit those noisy points. So which ensemble is more noise-robust depends on
the dataset and on how each is configured — on breast-cancer the ranking goes the
other way. "The forest beats AdaBoost under noise" is **dataset-dependent, not a
law**, and this chapter will not ship it as one. What *is* robust is the mechanism we
measured: the exponential loss concentrates weight on the hardest points. Know the
mechanism, not the slogan.


## Error analysis, honest limits, and the bridge to chapter 08

AdaBoost on spam is **strong**: 0.949, competitive with the forest, resistant to
extra rounds on clean data, and — on this problem — more robust to label noise than
the forest. Stated with the same honesty:

- It did **not win** — the forest edged it by a point. There is no universally best
  model; the family that wins tracks the shape of the problem.
- Its resistance is **not immunity**. Under heavy label noise more rounds hurt; the
  fix is early stopping or a gentler loss, not more rounds.
- Its importances reflect **this corpus**, not cause (`george`, `hp`).
- It can only be as good as what the stump base can separate — the training error
  floored at 0.045, the irreducible overlap of these 57 features.

**When to push further.** AdaBoost is one specific choice — the exponential loss —
inside a much larger idea. **Chapter 08, gradient boosting**, reveals AdaBoost as a
special case and generalises it to *any* differentiable loss (including losses far
less savage on mislabeled points than the exponential) by doing gradient descent in
function space. That is the natural next step when AdaBoost's noise-sensitivity, or
its fixed loss, becomes the thing in your way.


## Your turn

1. **(warm-up)** From Figure C, read off the round at which the clean test error
   bottoms. In one sentence, say why running all the way to 400 rounds does not hurt
   here.
2. **(stretch)** Raise the injected noise in Figure F to 0.5 and re-measure the round
   at which AdaBoost's test error turns upward. Does the turn come *earlier* than at
   40 %? Relate your answer to the weight-concentration panel on the left.
3. **(deeper)** Swap the stump base for `DecisionTreeClassifier(max_depth=3)` and
   re-run the degradation curve (Figure G). Does the stronger base overfit the noise
   *faster* (NB 3 and NB 4), and does AdaBoost still beat the forest at 40 % noise?
   Explain from the bias/variance picture of the base learner.


## What you built

You ran an honest AdaBoost workflow on a real, 57-feature problem, end to end:

- It **shines** — 0.949, competitive with the random forest, far above a single
  stump — the chapter's promise, realised on real data.
- You read its **staged resistance** truthfully: the test error bottoms near round
  280 and stays flat, while the training error floors at 0.045 and never reaches zero
  (real data has irreducible overlap; "train error to zero" was a toy phenomenon).
- You read its **importances** honestly: genuine spam markers (`!`, `$`, `remove`)
  mixed with corpus artifacts (`george`, `hp`), MDI cross-checked against permutation,
  none of it causal.
- You understood its **noise behaviour from the inside**: the exponential loss piles
  weight onto the mislabeled points (~21 % of points, ~40 % of the weight), so under
  heavy noise more rounds overfit.
- And you **refused to over-generalise**: "the forest beats AdaBoost under noise"
  reverses on spam, so it is dataset-dependent, not a law.

**Vocabulary.** staged resistance · irreducible error / training-error floor ·
false-alarm asymmetry · MDI vs permutation importance · corpus artifact ·
exponential-loss non-robustness · weight concentration · no universal law.


## Chapter wrap — AdaBoost, end to end

Five notebooks, one method: reweighting the mistakes by hand (NB 1) → the additive
vote and where the weight alpha comes from, the exponential loss (NB 2) → rounds
versus learning rate and the two faces of overfitting (NB 3) → the estimator and its
dials, the base must stay weak (NB 4) → a demanding real case where it shines, with
its honest noise signature (NB 5).

AdaBoost is the course's first **boosting** method: weak learners trained in
*sequence*, each focused on the running ensemble's mistakes, combined as a weighted
vote — the exact opposite of chapter 06's *parallel* bagging, and built on chapter
04's decision stump. Its exponential loss is both its engine and its
noise-sensitivity. **Chapter 08, gradient boosting**, generalises it to any
differentiable loss and opens the most productive branch of modern tabular learning
(then XGBoost, LightGBM in ch 09–10).

**Going further (optional).** Choose the number of rounds with a validation curve
(early stopping); try SAMME with a slightly stronger base and watch the noise
trade-off; preview a robust loss in chapter 08.

**References.**
- Freund Y, Schapire R (1997). A decision-theoretic generalization of on-line
  learning. *J. Comput. Syst. Sci.* 55(1):119–139. doi:10.1006/jcss.1997.1504
- Schapire R, Freund Y, Bartlett P, Lee W (1998). Boosting the margin. *Ann. Statist.*
  26(5):1651–1686. doi:10.1214/aos/1024691352
- Friedman J, Hastie T, Tibshirani R (2000). Additive logistic regression: a
  statistical view of boosting. *Ann. Statist.* 28(2):337–407. doi:10.1214/aos/1016218223
- Zhu J, Zou H, Rosset S, Hastie T (2009). Multi-class AdaBoost (SAMME). *Stat.
  Interface* 2(3):349–360. doi:10.4310/SII.2009.v2.n3.a8
- Dietterich T (2000). An experimental comparison of bagging, boosting, and
  randomization. *Mach. Learn.* 40(2):139–157. doi:10.1023/A:1007607513941
- Hopkins M, Reeber E, Forman G, Suermondt J. Spambase. UCI Machine Learning
  Repository. doi:10.24432/C53G6X
- Hastie T, Tibshirani R, Friedman J. *The Elements of Statistical Learning*, ch 10.
  doi:10.1007/978-0-387-84858-7
- James G, Witten D, Hastie T, Tibshirani R. *An Introduction to Statistical
  Learning*, §8.2. doi:10.1007/978-1-0716-1418-1

*Previous: 04 — The estimator AdaBoostClassifier and its parameters.*
*Next: Module 08 — Gradient Boosting.*
